# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ShamKottish/FlyRankML/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

Lane: CTR / Engagement Opportunity Scoring

I chose this lane because the project can help identify visible pages that receive fewer clicks or weaker engagement than comparable pages. The unit of analysis is one pseudonymized content item. Instead of comparing every page using one CTR threshold, I will compare pages with others in the same position tier because ranking position strongly affects CTR. The intended output is a ranked review queue showing which pages should be inspected first and why.

In [9]:
import os
import sys
import subprocess

IN_COLAB = "google.colab" in sys.modules

REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, REPO_DIR],
            check=True
        )

    os.chdir(REPO_DIR)

    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"],
        check=True
    )

print("Working directory:", os.getcwd())

assert os.path.exists(
    "data/raw/content_refresh_anonymized.csv"
), "Starter CSV still not found."

print(" Starter data found. Ready to continue.")

Working directory: /content/flyrank-ml-internship-starter/flyrank-ml-internship-starter
 Starter data found. Ready to continue.


## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*


The decision is which content items an SEO specialist or content editor should inspect first. Depending on the reason code, the reviewer could examine the title and metadata, check whether the page matches search intent, improve the page structure and content, or decide to monitor it without making a change.

A false positive would waste editorial time and could lead someone to change a page that did not need intervention. A false negative would miss a potentially valuable improvement opportunity. Data can help because CTR depends strongly on ranking position, traffic volume, content type, age, and engagement. I will begin with a transparent position-adjusted rule and only use a more complex model later if it improves the ranked queue. This is provisionally a ranking/scoring task, evaluated using precision@K and a manual review of the highest-ranked items.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

MIN_IMPRESSIONS = 500
MIN_SESSIONS = 50
MIN_CTR_GAP_PP = 0.10
LOW_ENGAGEMENT_PCT = 30

decision_settings = {
    "unit_of_analysis": "one pseudonymized content item",
    "output": "ranked review queue",
    "minimum_impressions": MIN_IMPRESSIONS,
    "minimum_sessions": MIN_SESSIONS,
    "minimum_CTR_gap_percentage_points": MIN_CTR_GAP_PP,
    "low_engagement_threshold_percent": LOW_ENGAGEMENT_PCT,
}

for setting, value in decision_settings.items():
    print(f"{setting}: {value}")


unit_of_analysis: one pseudonymized content item
output: ranked review queue
minimum_impressions: 500
minimum_sessions: 50
minimum_CTR_gap_percentage_points: 0.1
low_engagement_threshold_percent: 30


## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*


The starter dataset contains 30,000 anonymized content items across 32 clients. After requiring at least 500 impressions and comparing each page with the median CTR of its own position tier, 3,140 pages, or 10.5% of the inventory, have a CTR gap greater than 0.10 percentage points. The data also contains 4,919 pages, or 16.4%, with at least 50 sessions but engagement below 30%. These observed groups are large enough to support a position-adjusted review-ranking project.

In [11]:
from pathlib import Path
import pandas as pd

# Load the starter dataset
possible_paths = [
    Path("data/raw/content_refresh_anonymized.csv"),
    Path("/content/flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv"),
]

DATA_PATH = next((p for p in possible_paths if p.exists()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        "Starter CSV not found. Run the Notebook 01 setup cell first."
    )

df = pd.read_csv(DATA_PATH)

# Thresholds used for this quick analysis
MIN_IMPRESSIONS = 500
MIN_SESSIONS = 50
MIN_CTR_GAP_PP = 0.10
LOW_ENGAGEMENT_PCT = 30

# CTR opportunity:
# compare pages only with others in the same position tier
ctr_base = df[
    (df["avg_position"] > 0) &
    (df["impressions_90d"] >= MIN_IMPRESSIONS)
].copy()

ctr_base["tier_median_ctr"] = (
    ctr_base.groupby("position_tier")["ctr"]
    .transform("median")
)

ctr_base["ctr_gap_pp"] = (
    ctr_base["tier_median_ctr"] - ctr_base["ctr"]
)

ctr_candidates = ctr_base[
    ctr_base["ctr_gap_pp"] > MIN_CTR_GAP_PP
].copy()

# Engagement opportunity
engagement_candidates = df[
    (df["sessions_90d"] >= MIN_SESSIONS) &
    (df["engagement_rate"] < LOW_ENGAGEMENT_PCT)
].copy()

# Pages showing both signals
both_candidate_ids = set(ctr_candidates["content_id"]).intersection(
    engagement_candidates["content_id"]
)

# Print the supporting numbers
print(
    f"Inventory: {len(df):,} pages across "
    f"{df['client_id'].nunique():,} clients"
)

print(
    f"CTR-gap candidates: {len(ctr_candidates):,} "
    f"({len(ctr_candidates) / len(df):.1%})"
)

print(
    f"Low-engagement candidates: {len(engagement_candidates):,} "
    f"({len(engagement_candidates) / len(df):.1%})"
)

print(
    f"Pages with both signals: {len(both_candidate_ids):,} "
    f"({len(both_candidate_ids) / len(df):.1%})"
)

Inventory: 30,000 pages across 32 clients
CTR-gap candidates: 3,140 (10.5%)
Low-engagement candidates: 4,919 (16.4%)
Pages with both signals: 427 (1.4%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

I can say that the dataset contains an observed group of high-impression pages whose CTR is lower than the median CTR of other pages in the same position tier. I can also say that some pages have meaningful session volume but measured engagement below the selected threshold. These are directional, decision-support findings that can be used to create a ranked human-review queue.

I cant claim that a weak title, metadata, or page structure caused the low CTR or engagement. I also cannot claim that editing a recommended page will increase its traffic, prove a Google ranking factor, reverse-engineer Google's algorithm, or establish a causal relationship. Any recommended page must still be inspected by a human before action is taken.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Confirm that the analysis does not rely on raw private fields.

raw_private_fields = {
    "client_name",
    "domain",
    "url",
    "raw_url",
    "query",
    "raw_query",
    "keyword_text",
    "content_title",
}

private_fields_present = sorted(raw_private_fields.intersection(df.columns))

print("Raw private fields present:", private_fields_present)

assert not private_fields_present, (
    "Unexpected raw or identifying columns were found."
)

assert (ctr_candidates["avg_position"] > 0).all()
assert (ctr_candidates["impressions_90d"] >= MIN_IMPRESSIONS).all()

print("Safety check passed.")
print("The analysis uses pseudonymized IDs and observed aggregate measurements.")
print("The results are decision-support findings, not causal proof.")

Raw private fields present: []
Safety check passed.
The analysis uses pseudonymized IDs and observed aggregate measurements.
The results are decision-support findings, not causal proof.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.